# SI4006 · Sesión 4 — Lab: Fine-tuning con LoRA
### Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT · Semana 4

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manularrea/EAFIT-SI4006/blob/main/sesiones/s04/S04_Lab_Fine_tuning.ipynb)

Hoy su modelo base deja de ser genérico y aprende **su** dominio.

El laboratorio tiene tres partes:
- **Lab A — Baseline:** medir qué tan bien hace la tarea el modelo *sin* entrenar.
- **Lab B — Fine-tuning con LoRA:** enseñarle con sus 20 ejemplos.
- **Lab C — Medir la mejora:** comparar contra el baseline.

> **Antes de empezar:** activen la GPU. `Runtime → Change runtime type → T4 GPU`. Sin GPU el entrenamiento tarda muchísimo.


## 0 · Preparación del entorno
Instalamos las librerías del ecosistema Hugging Face que vimos en clase.

In [ ]:
# Instalación (silenciosa). Puede tardar ~1 minuto.
!pip install -q transformers datasets peft accelerate evaluate bitsandbytes 2>/dev/null
print("Librerías instaladas.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Verificar que hay GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cpu":
    print("⚠️  No hay GPU activa. Runtime → Change runtime type → T4 GPU, y reinicien.")

## 1 · Su modelo base
En la Sesión 3 cada equipo eligió un modelo. Pónganlo aquí.

Si aún no tienen uno decidido, el default (`Qwen2.5-0.5B`) es pequeño, en español y entrena rápido en Colab — sirve para aprender el pipeline.

In [ ]:
# Modelo base. Cámbienlo por el de su equipo.
MODEL_ID = "Qwen/Qwen2.5-0.5B"   # decoder pequeño, multilingüe, entrena en minutos

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# TODO 1 — carguen el modelo con AutoModelForCausalLM.from_pretrained
#          Pistas: pásenle MODEL_ID, torch_dtype=torch.float16, y muévanlo a .to(device)
model = None

print(f"Modelo cargado: {MODEL_ID}")

---
## Lab A · El baseline
**La pieza más importante de M1.** Antes de entrenar, medimos qué tan bien hace el modelo la tarea. Este número es su punto de comparación: sin él, no pueden demostrar que el fine-tuning sirvió.

Definan una función que le pregunte algo al modelo y devuelva su respuesta.

In [ ]:
def preguntar(prompt, max_new=80):
    """Le da un prompt al modelo y devuelve su respuesta."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new,
                             do_sample=False, pad_token_id=tokenizer.pad_token_id)
    texto = tokenizer.decode(out[0], skip_special_tokens=True)
    return texto[len(prompt):].strip()

# TODO 2 — prueben el baseline con una pregunta de SU dominio.
#          Guarden la respuesta: la van a comparar al final.
pregunta = "..."   # cámbienla por algo de su dominio
print("PREGUNTA:", pregunta)
print("BASELINE:", preguntar(pregunta))

**Anoten esta respuesta.** Es cómo responde el modelo *antes* de conocer su dominio. Al final del laboratorio van a comparar contra ella.

---
## 2 · Sus datos
Aquí van los **20 ejemplos** que trajeron. Cada uno es un par `entrada → salida deseada`.

Reemplacen los ejemplos de abajo por los de su dominio. Mientras más representativos, mejor aprende el modelo.

In [ ]:
# Sus ejemplos: pares (entrada, salida deseada).
# Reemplacen por los 20 de su equipo. Aquí van 3 de muestra (dominio educación).
ejemplos = [
    {"entrada": "Explica qué es una fracción.",
     "salida": "Una fracción representa partes de un todo. Por ejemplo, 1/4 es una de cuatro partes iguales."},
    {"entrada": "¿Qué es un número primo?",
     "salida": "Un número primo solo se puede dividir exactamente entre 1 y él mismo, como 2, 3, 5 o 7."},
    {"entrada": "Explica la suma de fracciones.",
     "salida": "Para sumar fracciones con el mismo denominador, se suman los numeradores y se mantiene el denominador."},
    # ... añadan aquí el resto de sus ejemplos ...
]
print(f"{len(ejemplos)} ejemplos cargados.")

Convertimos los ejemplos al formato que el modelo espera: un solo texto por ejemplo, con la entrada y la salida juntas.

In [ ]:
from datasets import Dataset

def formatear(ej):
    texto = f"Pregunta: {ej['entrada']}\nRespuesta: {ej['salida']}{tokenizer.eos_token}"
    return {"text": texto}

dataset = Dataset.from_list([formatear(e) for e in ejemplos])

def tokenizar(batch):
    out = tokenizer(batch["text"], truncation=True, max_length=256, padding="max_length")
    # TODO 3 — en Causal LM, las etiquetas (labels) son una copia de los input_ids.
    #          Añadan out["labels"] = ...
    return out

dataset = dataset.map(tokenizar, remove_columns=["text"])
print("Dataset listo:", dataset)

---
## Lab B · Fine-tuning con LoRA
Configuramos LoRA con la librería `peft`. Recuerden los tres números de clase: **rank**, **alpha**, **target_modules**.

In [ ]:
from peft import LoraConfig, get_peft_model

# TODO 4 — completen la configuración de LoRA con los valores de clase.
#          rank=8, alpha=16, target_modules=["q_proj","v_proj"], task_type="CAUSAL_LM"
lora_config = LoraConfig(
    r=None,              # <- rank
    lora_alpha=None,     # <- alpha
    target_modules=None, # <- capas de atención
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # deben ver <1% entrenable

Ahora el entrenamiento con la `Trainer` API que vimos en las diapositivas.

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="./lora-out",
    learning_rate=2e-4,
    num_train_epochs=10,          # con pocos ejemplos, varias épocas ayudan
    per_device_train_batch_size=2,
    logging_steps=5,
    report_to="none",             # cámbienlo a "wandb" si configuraron W&B
    fp16=True,
)

trainer = Trainer(model=model, args=args, train_dataset=dataset)
trainer.train()
print("Entrenamiento terminado.")

> **Miren la columna `Training Loss`.** Debe bajar época tras época. Esa curva descendente es la señal de que el modelo está aprendiendo sus ejemplos — lo mismo que verían en W&B.

---
## Lab C · ¿Mejoró?
El momento de la verdad: le hacemos **la misma pregunta** del baseline, ahora con el modelo entrenado, y comparamos.

In [ ]:
# TODO 5 — hagan la MISMA pregunta del Lab A, ahora con el modelo entrenado.
#          Usen la función preguntar() otra vez.
print("PREGUNTA:", pregunta)
print()
print("DESPUÉS del fine-tuning:")
# print(...)
print()
print("Compárenla con el baseline que anotaron al principio.")

### Para su entrega M1
Documenten en una celda de texto:
1. La pregunta que usaron.
2. La respuesta **baseline** (antes).
3. La respuesta **después** del fine-tuning.
4. Su lectura: ¿mejoró? ¿en qué? ¿por qué creen que sí o que no?

Eso, junto con este notebook ejecutado y sus datos documentados, **es su entrega M1**.

## 3 · Guardar el modelo
LoRA guarda solo las matrices nuevas: unos pocos megabytes, no el modelo entero.

In [ ]:
model.save_pretrained("./mi-modelo-lora")
print("Guardado. El adaptador LoRA pesa solo unos MB — esa es la gracia de LoRA.")

# Para subirlo al Hub (opcional):
# model.push_to_hub("su-usuario/su-modelo")

---
### Cierre
Acaban de tomar un modelo genérico y enseñarle su dominio. Eso es fine-tuning, y es la base de casi todo sistema de IA especializado que existe.

**Su entrega M1** sale de este notebook: el pipeline ejecutado, sus datos documentados, y la comparación baseline vs. resultado.

Nos vemos en la Sesión 5 — donde dejamos de preguntar *"¿entrena?"* y empezamos a preguntar *"¿es bueno de verdad?"*.
